# الذكاء الاصطناعي في الهندسة المدنية — الفصل 6 Deep learning (b)
Companion notebook. Run all cells (Runtime → Run all). Every number in the book comes from this code.

In [ ]:
!pip -q install xgboost shap scikit-image statsmodels ucimlrepo
import os; os.makedirs('fig', exist_ok=True)

In [ ]:
%%writefile crackgen.py
import numpy as np
from scipy.ndimage import gaussian_filter
def concrete_bg(rng, n=64):
    img = 0.55 + 0.08*gaussian_filter(rng.normal(0,1,(n,n)), 1.2) + 0.03*rng.normal(0,1,(n,n))
    for _ in range(rng.integers(3,12)):                      # pores
        y,x = rng.integers(0,n,2); r=rng.uniform(0.6,1.6)
        yy,xx=np.ogrid[:n,:n]; img[(yy-y)**2+(xx-x)**2<=r*r] -= rng.uniform(0.1,0.25)
    gy,gx = np.mgrid[:n,:n]/n; img += rng.uniform(-0.12,0.12)*gx + rng.uniform(-0.12,0.12)*gy  # lighting
    if rng.random()<0.3:                                      # shadow band
        a=rng.uniform(0,np.pi); d=(gx-0.5)*np.cos(a)+(gy-0.5)*np.sin(a); img[d>rng.uniform(-0.3,0.3)] -= rng.uniform(0.08,0.18)
    return img
def draw_path(img, pts, width, depth):
    n=img.shape[0]; yy,xx=np.mgrid[:n,:n]; m=np.zeros_like(img)
    for (y0,x0),(y1,x1) in zip(pts[:-1],pts[1:]):
        for t in np.linspace(0,1,8):
            y=y0+(y1-y0)*t; x=x0+(x1-x0)*t
            m=np.maximum(m, np.exp(-((yy-y)**2+(xx-x)**2)/(2*(width/2)**2)))
    return img - depth*m, m>0.5
def crack(img, rng):
    n=img.shape[0]; y,x=rng.uniform(16,48,2); a=rng.uniform(0,2*np.pi); pts=[(y,x)]
    for _ in range(rng.integers(8,16)):
        a+=rng.normal(0,0.6); y+=3.5*np.sin(a); x+=3.5*np.cos(a)
        if not (4<y<60 and 4<x<60): a+=np.pi; y=np.clip(y,4,60); x=np.clip(x,4,60)
        pts.append((y,x))
    return draw_path(img, pts, rng.uniform(0.8,2.6), rng.uniform(0.15,0.45))
def joint(img, rng):                                           # straight formwork joint = hard negative
    n=img.shape[0]; a=rng.uniform(0,np.pi); c=rng.uniform(16,48)
    p0=(c-40*np.sin(a), c-40*np.cos(a)); p1=(c+40*np.sin(a), c+40*np.cos(a))
    return draw_path(img,[p0,p1],rng.uniform(1.0,2.2),rng.uniform(0.15,0.4))
def make_images(N, seed=0, joints=True):
    rng=np.random.default_rng(seed); X=np.zeros((N,64,64),np.float32); y=np.zeros(N,np.int64); M=np.zeros((N,64,64),bool)
    for i in range(N):
        img=concrete_bg(rng)
        if rng.random()<0.5: img,M[i]=crack(img,rng); y[i]=1
        elif joints and rng.random()<0.4: img,_=joint(img,rng)
        X[i]=np.clip(img,0,1)
    return X,y,M


In [ ]:
import numpy as np, torch, torch.nn as nn
from crackgen import make_images
import importlib.util
torch.manual_seed(0); torch.set_num_threads(2)
norm=lambda X:(X-X.mean(axis=(1,2),keepdims=True))/(X.std(axis=(1,2),keepdims=True)+1e-6)
class CNN(nn.Module):
    def __init__(s):
        super().__init__()
        s.f=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
                          nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
                          nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.AdaptiveMaxPool2d(1))
        s.h=nn.Linear(64,2)
    def forward(s,x): return s.h(s.f(x).flatten(1))
def train(X,y,ep=6):
    m=CNN(); opt=torch.optim.Adam(m.parameters(),2e-3); lf=nn.CrossEntropyLoss()
    Xt=torch.tensor(norm(X))[:,None]; Yt=torch.tensor(y)
    for e in range(ep):
        m.train(); perm=torch.randperm(len(Xt))
        for b in range(0,len(Xt),64):
            i=perm[b:b+64]; opt.zero_grad(); lf(m(Xt[i]),Yt[i]).backward(); opt.step()
    return m.eval()
# test set with joints, tracking which negatives contain joints
rng_seed=5
from crackgen import concrete_bg, crack, joint
rng=np.random.default_rng(rng_seed); N=1000; X=np.zeros((N,64,64),np.float32); y=np.zeros(N,int); J=np.zeros(N,bool)
for i in range(N):
    img=concrete_bg(rng)
    if rng.random()<0.5: img,_=crack(img,rng); y[i]=1
    elif rng.random()<0.4: img,_=joint(img,rng); J[i]=True
    X[i]=np.clip(img,0,1)
for name,jflag in [("trained WITHOUT joints",False),("trained WITH joints",True)]:
    Xtr,ytr,_=make_images(4000,1,joints=jflag); m=train(Xtr,ytr)
    with torch.no_grad(): p=m(torch.tensor(norm(X))[:,None]).argmax(1).numpy()
    acc=(p==y).mean(); fp_joint=p[J].mean(); fp_plain=p[(y==0)&~J].mean(); rec=p[y==1].mean()
    print(f"{name}: acc={acc:.3f} recall={rec:.3f} false-alarm on joints={fp_joint:.3f} on plain={fp_plain:.3f} (joint imgs={J.sum()})")